# Benchmarky a vyhodnocení modelu

V tomto notebooku navazujeme na předchozí zpracování dat. Načteme si rozdělené sady (Train a Validation), aplikujeme identický preprocessing jako v prvním kroku a stanovíme základní výkonnostní metriky pomocí naivního modelu a logistické regrese.

In [5]:
import pandas as pd
import numpy as np
import os
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, accuracy_score, classification_report, brier_score_loss

# --- 1. PŘÍPRAVA SLOŽEK A STAŽENÍ DAT ---
# Vytvoření složky 'data' pro splnění strukturních požadavků HW2
os.makedirs('data', exist_ok=True)

# ID tvých souborů z Google Drive
file_ids = {
    'train.csv': '1pIaKZQho-MVPGRrcJ6_ECA18TOjdir_h',
    'validation.csv': '1CoJMwK2MeFe_jiMMHnfWB9g0xB2NQPwg',
    'test.csv': '1di85Mr_ZQAVJtP823Cy8y6WyqI_mhRah'
}

print("Stahuji rozdělené datové sady z Google Drive...")
for filename, file_id in file_ids.items():
    !gdown --id {file_id} -O data/{filename}

# --- 2. NAČTENÍ DAT ---
# Načítáme přímo hotové sady, čímž eliminujeme riziko chyby při opakovaném splitu[cite: 5]
df_train = pd.read_csv('data/train.csv')
df_val = pd.read_csv('data/validation.csv')

# Převod na datetime pro následný feature engineering
df_train['launched'] = pd.to_datetime(df_train['launched'])
df_train['deadline'] = pd.to_datetime(df_train['deadline'])
df_val['launched'] = pd.to_datetime(df_val['launched'])
df_val['deadline'] = pd.to_datetime(df_val['deadline'])

print(f"\nData připravena. Train: {len(df_train)} řádků, Val: {len(df_val)} řádků.")

Stahuji rozdělené datové sady z Google Drive...
/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From: https://drive.google.com/uc?id=1pIaKZQho-MVPGRrcJ6_ECA18TOjdir_h
To: /content/data/train.csv
100% 14.4M/14.4M [00:00<00:00, 77.5MB/s]
/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From: https://drive.google.com/uc?id=1CoJMwK2MeFe_jiMMHnfWB9g0xB2NQPwg
To: /content/data/validation.csv
100% 3.09M/3.09M [00:00<00:00, 187MB/s]
/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a f

In [6]:
# --- 3. REPRODUKCE PREPROCESSINGU ---
def add_features(data):
    """Odvození délky kampaně (pre-launch feature)[cite: 3]"""
    d = data.copy()
    d['duration_days'] = (d['deadline'] - d['launched']).dt.days
    return d

X_train_raw = add_features(df_train)
X_val_raw = add_features(df_val)

num_cols = ['usd_goal_real', 'duration_days']
cat_cols = ['main_category', 'country']

y_train = X_train_raw['target']
X_train = X_train_raw[num_cols + cat_cols]
y_val = X_val_raw['target']
X_val = X_val_raw[num_cols + cat_cols]

# Nastavení pipeline (Standardizace + One-Hot Encoding)[cite: 3]
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols)
    ])

# ZLATÉ PRAVIDLO: Fitujeme pouze na trénovací sadě (prevence leakage)
preprocessor.fit(X_train)
X_train_processed = preprocessor.transform(X_train)
X_val_processed = preprocessor.transform(X_val)

# --- 4. BENCHMARKY ---
# A. Naivní baseline (předpovídá vždy neúspěch - většinová třída 0)[cite: 1]
baseline_preds = np.zeros(len(y_val))
baseline_probs = np.zeros(len(y_val))

# B. Jednoduchý ML benchmark (Logistická regrese)[cite: 1, 2]
log_reg = LogisticRegression(max_iter=1000, random_state=42)
log_reg.fit(X_train_processed, y_train)

ml_preds = log_reg.predict(X_val_processed)
ml_probs = log_reg.predict_proba(X_val_processed)[:, 1]

# --- 5. VÝSLEDKY ---
results = pd.DataFrame({
    'Metrika': ['Accuracy', 'F1-Score', 'Brier Score'],
    'Naivní Baseline': [
        accuracy_score(y_val, baseline_preds),
        f1_score(y_val, baseline_preds, zero_division=0),
        brier_score_loss(y_val, baseline_probs)
    ],
    'Logistická Regrese': [
        accuracy_score(y_val, ml_preds),
        f1_score(y_val, ml_preds),
        brier_score_loss(y_val, ml_probs)
    ]
})

print("\n--- POROVNÁNÍ BENCHMARKŮ NA VALIDAČNÍ SADĚ ---")
display(results.round(4))

print("\n--- DETAILNÍ REPORT (Logistická regrese) ---")
print(classification_report(y_val, ml_preds))


--- POROVNÁNÍ BENCHMARKŮ NA VALIDAČNÍ SADĚ ---


,Metrika,Naivní Baseline,Logistická Regrese
0,Accuracy,0.6259,0.6586
1,F1-Score,0.0000,0.3965
2,Brier Score,0.3741,0.2140



--- DETAILNÍ REPORT (Logistická regrese) ---
              precision    recall  f1-score   support

           0       0.68      0.87      0.76     31138
           1       0.59      0.30      0.40     18613

    accuracy                           0.66     49751
   macro avg       0.63      0.59      0.58     49751
weighted avg       0.64      0.66      0.63     49751



## 9. Krátké shrnutí

---

### **Klíčové metodické body**
Na základě zpětné vazby z DÚ1 jsme implementovali postupy pro zajištění maximální robustnosti modelu:

*   **Prevence Time Leakage:** Použili jsme striktní chronologický **time-based split** podle data `launched`.
*   **Obrana proti Concept Driftu:** Tímto rozdělením jsme zajistili, že model testujeme na schopnost generalizace v čase, což simuluje reálné nasazení.
*   **Konzistence datové pipeline:** V souladu se zadáním pracujeme s dříve uloženými a fixními datovými sadami (`train.csv`, `validation.csv`, `test.csv`).
    > Tento postup eliminuje riziko nechtěných změn v rozdělení dat při opakovaném spouštění výpočtů a zaručuje reprodukovatelnost výsledků.
*   **Integrita preprocessingu:** Veškeré transformace (scaling, encoding) byly učeny (**fit**) výhradně na trénovací sadě. Tím jsme eliminovali riziko **preprocessing leakage**.
*   **Pre-launch focus:** Model pracuje striktně pouze s příznaky (features), které jsou tvůrci projektu známy ještě před ostrým startem kampaně.

### **Analytické výzvy a insights**
Nejobtížnější částí bylo vyladit metriky tak, aby odpovídaly **cost-sensitive** povaze byznys zadání.

> **Hlavní zjištění:** Protože chyba typu **False Positive** (falešný příslib úspěchu) představuje pro tvůrce nejvyšší riziko zbytečné investice, rozšířili jsme evaluaci o **Brier Score**.

*   Tato metrika nám pomáhá hodnotit kvalitu **kalibrace pravděpodobností**.
*   Cílem je vracet tvůrci spolehlivý odhad šance na úspěch, nikoliv jen striktní binární rozhodnutí.

### **Další kroky (Next Steps)**
V navazující fázi projektu se zaměříme na:

1.  **Pokročilé modely:** Experimentování se složitějšími algoritmy jako **Random Forest** nebo **XGBoost**.
2.  **Rozšířený Feature Engineering:** Vytvoření nových časových příznaků, například sezónnost nebo vliv konkrétního měsíce spuštění.
3.  **Thresholding:** Explicitní práce s posouváním rozhodovacího prahu pro aktivní potlačení drahých **False Positive** chyb.

---